# OpenAI Function Calling

In [ ]:
# 导入基础库：os 用于读取环境变量，openai 是官方 SDK
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())  # 读取本地 .env 文件，把里面的环境变量加载进 os.environ
openai.api_key = os.environ['OPENAI_API_KEY']

# 【版本兼容】openai>=1.0 之后不再用全局的 openai.ChatCompletion.create() 方式调用接口，
# 而是要先创建一个 client 对象，再用 client.chat.completions.create(...) 调用。
# 下面这个 client 会在后续所有请求中复用。
client = openai.OpenAI(api_key=openai.api_key)

In [ ]:
import json

# 这是一个"假的"天气查询函数，写死返回固定结果，用来模拟真实的后端 API 或第三方接口。
# 在 function calling 的场景里，模型本身不会真的执行这个函数——
# 它只会"决定"要调用它、并生成调用参数，真正执行是由我们这段 Python 代码完成的。
def get_current_weather(location, unit="fahrenheit"):
    """Get the current weather in a given location"""
    weather_info = {
        "location": location,
        "temperature": "72",
        "unit": unit,
        "forecast": ["sunny", "windy"],
    }
    return json.dumps(weather_info)  # 返回 JSON 字符串，方便作为 "function" 角色消息的 content

In [ ]:
# 定义函数的 JSON Schema，描述给模型看，让模型知道"有哪些函数可以调用、参数长什么样"。
# 这是旧版 function calling 的写法（用顶层的 "functions" 参数，元素里直接写 name/description/parameters）。
# 新版 OpenAI API 推荐用 "tools" 参数（每个元素外面包一层 {"type": "function", "function": {...}}），
# 但旧的 "functions" 参数目前仍被兼容保留，这里保持课程原本的写法方便理解概念。
functions = [
    {
        "name": "get_current_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA",
                },
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["location"],  # 只有 location 是必填参数，unit 可选
        },
    }
]

In [ ]:
# 构造对话消息列表，role="user" 表示这是用户说的话
messages = [
    {
        "role": "user",
        "content": "What's the weather like in Boston?"
    }
]

In [ ]:
# 注：openai 在第一个 cell 已经 import 过了，这里重复 import 不会报错，只是多余，保留原notebook结构不做删除
import openai

In [ ]:
# 调用 Chat Completion 接口，把 functions 传进去，模型会自己判断要不要调用某个函数
# 【版本兼容修复】原来的 openai.ChatCompletion.create(...) 是 openai<1.0 的写法，
# 在新版 SDK 里 openai.ChatCompletion 已经被移除（会直接抛 APIRemovedInV1 异常），
# 必须改成用前面创建好的 client 对象：client.chat.completions.create(...)
response = client.chat.completions.create(
    # OpenAI Updates: As of June 2024, we are now using the GPT-3.5-Turbo model
    model="gpt-3.5-turbo",
    messages=messages,
    functions=functions
)

In [ ]:
# 打印完整响应对象；新版 SDK 里 response 是一个 pydantic 模型（不是 dict），
# 打印出来会显示成 ChatCompletion(...) 的形式，但内容信息是一样的
print(response)

In [ ]:
# 取出模型返回的第一条消息
# 【版本兼容修复】旧版 response 是 dict，可以用 response["choices"][0]["message"] 取值；
# 新版 SDK 的 response 是 pydantic 对象，不支持下标访问，要用属性访问 response.choices[0].message
response_message = response.choices[0].message

In [ ]:
# 查看这条消息本身的结构（role、content、function_call 等字段）
response_message

In [ ]:
# content 字段：如果模型选择调用函数而不是直接回答文字，这里通常是 None
# 【版本兼容修复】改成属性访问 .content，而不是 ["content"]
response_message.content

In [ ]:
# function_call 字段：模型认为应该调用哪个函数、以及生成的参数（还是 JSON 字符串形式）
# 【版本兼容修复】改成属性访问 .function_call
response_message.function_call

In [ ]:
# function_call.arguments 是一个 JSON 字符串，需要用 json.loads 解析成 Python dict 才能用
# 【版本兼容修复】.function_call.arguments 属性访问，而不是 ["function_call"]["arguments"]
json.loads(response_message.function_call.arguments)

In [ ]:
# 把解析出来的参数保存到 args 变量里，后面要用它去真正调用 get_current_weather 函数
args = json.loads(response_message.function_call.arguments)

In [ ]:
# 【真实 bug 修复】args 是一个 dict，例如 {"location": "Boston, MA"}。
# 原代码写的是 get_current_weather(args)，相当于把整个 dict 当成 location 位置参数传进去，
# 参数没有被正确展开，是传参方式错误（typo）。
# 正确写法要用 ** 把 dict 展开成关键字参数：location=..., unit=...
get_current_weather(**args)

In [ ]:
# 换一条跟天气完全无关的用户消息，测试模型在"不需要调用函数"时的行为
messages = [
    {
        "role": "user",
        "content": "hi!",
    }
]

In [ ]:
# 依然把 functions 传进去，但因为消息内容和天气无关，模型应该会选择直接回复文字而不调用函数
# 【版本兼容修复】client.chat.completions.create 替代旧版 openai.ChatCompletion.create
response = client.chat.completions.create(
    # OpenAI Updates: As of June 2024, we are now using the GPT-3.5-Turbo model
    model="gpt-3.5-turbo",
    messages=messages,
    functions=functions,
)

In [ ]:
print(response)

In [ ]:
# function_call="auto"：让模型自己决定要不要调用函数（这也是默认行为）
messages = [
    {
        "role": "user",
        "content": "hi!",
    }
]
response = client.chat.completions.create(
    # OpenAI Updates: As of June 2024, we are now using the GPT-3.5-Turbo model
    model="gpt-3.5-turbo",
    messages=messages,
    functions=functions,
    function_call="auto",
)
print(response)

In [ ]:
# function_call="none"：强制模型不调用任何函数，只能用文字回复
messages = [
    {
        "role": "user",
        "content": "hi!",
    }
]
response = client.chat.completions.create(
    # OpenAI Updates: As of June 2024, we are now using the GPT-3.5-Turbo model
    model="gpt-3.5-turbo",
    messages=messages,
    functions=functions,
    function_call="none",
)
print(response)

In [ ]:
# 即使问的是天气（本该触发函数调用），只要 function_call="none"，模型也只会用文字回答（可能会编造答案）
messages = [
    {
        "role": "user",
        "content": "What's the weather in Boston?",
    }
]
response = client.chat.completions.create(
    # OpenAI Updates: As of June 2024, we are now using the GPT-3.5-Turbo model
    model="gpt-3.5-turbo",
    messages=messages,
    functions=functions,
    function_call="none",
)
print(response)

In [ ]:
# function_call={"name": "get_current_weather"}：强制模型必须调用这个指定的函数，
# 即使用户消息（"hi!"）跟天气完全无关，模型也会被迫生成调用参数（可能会瞎编 location）
messages = [
    {
        "role": "user",
        "content": "hi!",
    }
]
response = client.chat.completions.create(
    # OpenAI Updates: As of June 2024, we are now using the GPT-3.5-Turbo model
    model="gpt-3.5-turbo",
    messages=messages,
    functions=functions,
    function_call={"name": "get_current_weather"},
)
print(response)

In [ ]:
# 这次问题和天气相关，同时又强制指定调用 get_current_weather，是最"正常"的强制调用场景
messages = [
    {
        "role": "user",
        "content": "What's the weather like in Boston!",
    }
]
response = client.chat.completions.create(
    # OpenAI Updates: As of June 2024, we are now using the GPT-3.5-Turbo model
    model="gpt-3.5-turbo",
    messages=messages,
    functions=functions,
    function_call={"name": "get_current_weather"},
)
print(response)

In [ ]:
# 把模型这一轮返回的消息，追加进对话历史里，为下一轮请求做铺垫（多轮 function calling 的关键步骤）
# 【版本兼容修复】旧版 response["choices"][0]["message"] 本来就是个普通 dict，可以直接 append。
# 新版 SDK 里它是 pydantic 消息对象，不能直接塞进下一次请求的 messages 列表（会校验失败），
# 需要用 .model_dump() 转成普通 dict 再 append
messages.append(response.choices[0].message.model_dump())

In [ ]:
# 解析模型给出的调用参数，然后真正执行本地函数，得到"观测结果" observation
# 【版本兼容修复】.function_call.arguments 属性访问，而不是旧的三层 [""][""][""]
# 【真实 bug 修复】get_current_weather(args) 少了 **，同上面一样要展开 dict 成关键字参数
args = json.loads(response.choices[0].message.function_call.arguments)
observation = get_current_weather(**args)

In [ ]:
# 把函数的执行结果作为一条 role="function" 的消息追加进对话历史，
# 这样模型下一轮就能"看到"函数的返回值，并据此生成最终的自然语言回答
messages.append(
        {
            "role": "function",
            "name": "get_current_weather",
            "content": observation,
        }
)

In [ ]:
# 最后再调用一次模型，这次不需要再传 functions（也可以传，不影响），
# 模型会基于完整的对话历史（用户问题 + 函数调用 + 函数返回结果）生成自然语言总结回答
# 【版本兼容修复】client.chat.completions.create 替代旧版 openai.ChatCompletion.create
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages,
)
print(response)